# 📡 Fetching The Doom Data

**Don't Look Up: Asteroid Impact Tracker**  
*Created by Abdullah Hasan Dafa (@hasandafa)*

---

## 🎬 What's Happening Here?

Okay, so here's the deal. NASA has been tracking space rocks that might want to say "hi" to Earth. Some very aggressive "hi"s.

We're pulling data from **3 NASA APIs** because apparently one source of existential dread wasn't enough.

### APIs We're Hitting:
1. **Close Approaches** (CAD) - Every asteroid that got too close for comfort (1900-2200)
2. **Sentry** - The official "maybe panic?" list (~700 risky asteroids)
3. **NeoWs** - Asteroid Tinder profiles (size, speed, orbital drama)

Let's go! 🚀

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import json
import yaml
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scripts.fetch_nasa_data import NASADataFetcher

plt.style.use('dark_background')
sns.set_palette('husl')

print("✅ All imports loaded!")
print("📡 Ready to fetch some doom...")

✅ All imports loaded!
📡 Ready to fetch some doom...


## 🔑 Step 1: Check Your API Key

Before we can talk to NASA, we need credentials. Think of it as showing ID at the "End of the World" nightclub.

In [2]:
try:
    # When running from notebooks/ folder, config is in parent directory
    config_path = Path('../config.yaml')
    
    if not config_path.exists():
        print("❌ config.yaml not found!")
        print(f"   Looking in: {config_path.absolute()}")
        print("\n💡 Setup:")
        print("   cd ..")
        print("   cp config.yaml.example config.yaml")
    else:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        
        print("✅ config.yaml loaded successfully")
        print(f"   Location: {config_path.absolute()}")
        
        # Get API key file path (relative to config.yaml location)
        api_key_file = config['nasa_api']['api_key_file']
        # API key is in same directory as config (parent directory)
        api_key_path = config_path.parent / api_key_file
        
        print(f"\n🔍 Looking for API key file: {api_key_file}")
        print(f"   Full path: {api_key_path.absolute()}")
        
        # Check if API key file exists
        if not api_key_path.exists():
            print(f"\n❌ {api_key_file} not found!")
            print("\n📝 Setup instructions:")
            print(f"   1. Create file in project root: {api_key_file}")
            print("   2. Get API key from: https://api.nasa.gov/")
            print("   3. Paste your API key into the file (just the key, no quotes)")
            print("   4. Save and re-run this cell")
            print(f"\n   Quick command:")
            print(f"   cd .. && echo 'YOUR_KEY_HERE' > {api_key_file}")
        else:
            # Read and validate API key
            with open(api_key_path, 'r') as f:
                api_key = f.read().strip()
            
            if not api_key:
                print(f"❌ {api_key_file} is empty!")
                print("   Add your NASA API key to the file")
            elif api_key == "YOUR_NASA_API_KEY_HERE":
                print("❌ Still using placeholder key!")
                print("   Replace with your actual API key from https://api.nasa.gov/")
            else:
                # Success!
                print(f"\n✅ API key found and loaded!")
                print(f"🔑 Key preview: {api_key[:10]}...{api_key[-5:]}")
                print(f"📏 Key length: {len(api_key)} characters")
                print("\n🚀 Setup complete! Ready to fetch data!")
                print("   Run the next cell to initialize the fetcher")

except yaml.YAMLError as e:
    print(f"❌ Error parsing config.yaml: {e}")
    print("\n💡 Your config.yaml might have syntax errors")
    print("   Try using the clean version from artifacts")
    print("   Or run: python scripts/verify_setup.py")

except Exception as e:
    print(f"❌ Unexpected error: {e}")
    print("\n💡 Debug info:")
    print(f"   Current directory: {Path.cwd()}")
    print(f"   Expected config at: {config_path.absolute() if 'config_path' in locals() else 'N/A'}")

✅ config.yaml loaded successfully
   Location: c:\Users\user\Portofolio\do-not-look-up\notebooks\..\config.yaml

🔍 Looking for API key file: nasa_api_key.txt
   Full path: c:\Users\user\Portofolio\do-not-look-up\notebooks\..\nasa_api_key.txt

✅ API key found and loaded!
🔑 Key preview: DeoS6aJPJs...fWdr3
📏 Key length: 40 characters

🚀 Setup complete! Ready to fetch data!
   Run the next cell to initialize the fetcher


## 📡 Step 2: Fetch Close Approach Data

This API gives us a timeline of every asteroid that decided to swing by Earth. 

**Date Range:** 2020-2100  
**Max Distance:** 0.2 AU (that's ~30 million km, still way too close tbh)

This might take a minute. Maybe scroll Twitter while we wait?

In [3]:
fetcher = NASADataFetcher()

ca_df = fetcher.fetch_close_approaches()

print(f"\n📊 Fetched {len(ca_df):,} close approach records")
print(f"📅 Date range: {ca_df['cd'].min()} to {ca_df['cd'].max()}")
print(f"\n💭 That's a lot of 'near misses'...")

✅ API key loaded from c:\Users\user\Portofolio\do-not-look-up\notebooks\..\nasa_api_key.txt
🔑 Key preview: DeoS6aJPJs...fWdr3
🚀 NASA Data Fetcher initialized!
📡 Ready to download some doom data...

📡 Fetching Close Approach Data...
(Every time an asteroid said 'hey there' to Earth)
✅ Fetched 89,227 close approaches
💾 Saved to data\raw\close_approaches_raw.json

📊 Fetched 89,227 close approach records
📅 Date range: 2020-Apr-01 01:06 to 2100-Sep-30 11:13

💭 That's a lot of 'near misses'...


In [4]:
print("\n🔍 First 5 records:")
ca_df.head()


🔍 First 5 records:


,des,orbit_id,jd,cd,dist,dist_min,dist_max,v_rel,v_inf,t_sigma_f,h,fullname
0,2020 AY1,24,2458849.537516193,2020-Jan-01 00:54,0.0211637904862387,0.0211636477425536,0.021163933229925,5.62142224642355,5.59898133866116,< 00:01,25.30,(2020 AY1)
1,2019 YK,11,2458849.587204797,2020-Jan-01 02:06,0.0361009647238789,0.0360768259080859,0.0361251034508466,7.35926278450408,7.34922690439657,< 00:01,24.10,(2019 YK)
2,2013 EC20,14,2458849.640980606,2020-Jan-01 03:23,0.162018693203057,0.160649787675396,0.163387466048371,2.79370137646343,2.78780852454193,18:21,29.,(2013 EC20)
3,2020 AM1,6,2458849.804441915,2020-Jan-01 07:18,0.159657033568606,0.159564733866164,0.159749333072113,4.1529384618897,4.14891797104044,00:03,24.7,(2020 AM1)
4,2020 AP3,4,2458849.967322953,2020-Jan-01 11:13,0.0167404854088158,0.0165847722715622,0.0168961658040387,5.19125028298636,5.16049919025748,00:05,26.6,(2020 AP3)


## 🎯 Step 3: Fetch Sentry Risk Table

Now for the spicy part: The **Sentry Risk Table**.

This is NASA's official list of asteroids that *might* hit us. Emphasis on *might*.

Think of it as the VIP section of space rocks. They're on the list for a reason.

In [6]:
# Preview Sentry data
sentry_df = fetcher.fetch_sentry_objects()

print(f"\n📊 Found {len(sentry_df):,} high-risk asteroids")

if len(sentry_df) > 0:
    print("\n🔍 Sentry objects preview:")
    print(sentry_df.head(10))
    
    print(f"\n📋 Columns in Sentry data:")
    for col in sentry_df.columns:
        print(f"   • {col}")
    
    # Show most concerning one
    print(f"\n💀 Most concerning asteroid:")
    print(f"   Name: {sentry_df.iloc[0]['des']}")
    print(f"   Full name: {sentry_df.iloc[0]['fullname']}")
    print(f"   Impact probability: {sentry_df.iloc[0]['ip']}")
    print(f"   Torino Scale: {sentry_df.iloc[0]['ts_max']}")
    print(f"   Palermo Scale: {sentry_df.iloc[0]['ps_max']}")
else:
    print("\n✅ No high-risk asteroids currently tracked!")


📡 Fetching Sentry Risk Table...
(The 'maybe panic?' list)
✅ Fetched 1,998 high-risk asteroids
   (Don't worry, 'high-risk' is relative)
💾 Saved to data\raw\sentry_objects_raw.json

📊 Found 1,998 high-risk asteroids

🔍 Sentry objects preview:
              v_inf         des      h      fullname    last_obs  \
0  23.7606234552547     1979 XB  18.54     (1979 XB)  1979-12-15   
1  15.5694051293592    2022 KK2  28.45    (2022 KK2)  2022-05-23   
2  1.35802744453748  2000 SG344  24.79  (2000 SG344)  2000-10-03   
3  11.4626328606267   2012 VS76  26.95   (2012 VS76)  2012-11-16   
4             20.20     2018 GN  26.19     (2018 GN)  2018-04-09   
5              8.55     2011 TO  26.32     (2011 TO)  2013-01-12   
6  7.21203952320587  2012 BA102  26.51  (2012 BA102)  2012-02-21   
7  21.9988508209515  2014 HN197  19.92  (2014 HN197)  2014-04-27   
8  12.3834503195676    2005 UL6  24.51    (2005 UL6)  2005-11-22   
9  22.0982648784577     2009 DV  24.86     (2009 DV)  2009-02-24   

        

In [7]:
print("\n🔍 Sentry objects preview:")
sentry_df.head(10)


🔍 Sentry objects preview:


,v_inf,des,h,fullname,last_obs,ip,n_imp,ts_max,ps_max,id,ps_cum,last_obs_jd,diameter,range
0,23.7606234552547,1979 XB,18.54,(1979 XB),1979-12-15,8.515158e-07,4,0,-3.00,bJ79X00B,-2.70,2444222.5,0.66,2056-2113
1,15.5694051293592,2022 KK2,28.45,(2022 KK2),2022-05-23,0.0001203297828,33,0,-5.79,bK22K02K,-5.59,2459722.5,0.0069,2060-2122
2,1.35802744453748,2000 SG344,24.79,(2000 SG344),2000-10-03,0.002743395186,300,0,-3.11,bK00SY4G,-2.77,2451820.5,0.037,2069-2122
3,11.4626328606267,2012 VS76,26.95,(2012 VS76),2012-11-16,1.9442009e-05,15,0,-6.06,bK12V76S,-5.75,2456247.5,0.014,2081-2120
4,20.20,2018 GN,26.19,(2018 GN),2018-04-09,3.772e-09,1,0,-8.95,bK18G00N,-8.95,2458217.5,0.02,2102-2102
5,8.55,2011 TO,26.32,(2011 TO),2013-01-12,2.937e-06,1,0,-6.16,bK11T00O,-6.16,2456304.5,0.018,2064-2064
6,7.21203952320587,2012 BA102,26.51,(2012 BA102),2012-02-21,1.22522146e-05,20,0,-6.28,bK12BA2A,-6.03,2455978.5,0.017,2103-2122
7,21.9988508209515,2014 HN197,19.92,(2014 HN197),2014-04-27,6.5351e-09,12,0,-5.99,bK14HJ7N,-5.51,2456774.5,0.35,2069-2118
8,12.3834503195676,2005 UL6,24.51,(2005 UL6),2005-11-22,9.90088e-08,6,0,-6.96,bK05U06L,-6.71,2453696.5,0.042,2068-2119
9,22.0982648784577,2009 DV,24.86,(2009 DV),2009-02-24,3.1018e-08,2,0,-7.38,bK09D00V,-7.25,2454886.5,0.036,2044-2102


## 🌌 Step 4: Fetch NeoWs Data

Finally, we get the **detailed profiles** from NeoWs (Near Earth Object Web Service).

This includes:
- Size (diameter in km)
- Speed (relative velocity)
- Orbital elements (the physics stuff)
- Whether it's classified as "Potentially Hazardous" (PHAs)

⚠️ **WARNING:** This fetches A LOT of data. Go make coffee. Seriously.

In [8]:
print("☕ Grab coffee. This takes 5-10 minutes...")
print("📡 Fetching detailed asteroid data from NeoWs API...")

neows_df = fetcher.fetch_neows_data()

print(f"\n✅ SUCCESS! Fetched {len(neows_df):,} asteroid profiles")
print(f"📈 That's more asteroids than Starbucks locations in the US")

☕ Grab coffee. This takes 5-10 minutes...
📡 Fetching detailed asteroid data from NeoWs API...

📡 Fetching NeoWs Data...
(The asteroid dating profiles)
📅 Fetching from 2020-01-01 to 2100-12-31
⏱️  This might take a while. Maybe grab a coffee?


Fetching NeoWs:  87%|████████▋ | 3698/4227 [6:39:42<57:10,  6.49s/it]  



✅ Fetched 245,280 asteroid records
💾 Saved to data\raw\neows_data_raw.json

✅ SUCCESS! Fetched 245,280 asteroid profiles
📈 That's more asteroids than Starbucks locations in the US


In [9]:
print("\n🔍 NeoWs data preview:")
neows_df.head()


🔍 NeoWs data preview:


,links,id,neo_reference_id,name,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,is_sentry_object,sentry_data
0,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3564720,3564720,(2011 HS60),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,21.34,{'kilometers': {'estimated_diameter_min': 0.14...,False,"[{'close_approach_date': '2020-01-01', 'close_...",False,NaN
1,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3591759,3591759,(2011 YE40),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,25.20,{'kilometers': {'estimated_diameter_min': 0.02...,False,"[{'close_approach_date': '2020-01-01', 'close_...",False,NaN
2,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3630817,3630817,(2013 EC20),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,29.00,{'kilometers': {'estimated_diameter_min': 0.00...,False,"[{'close_approach_date': '2020-01-01', 'close_...",True,http://api.nasa.gov/neo/rest/v1/neo/sentry/363...
3,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3747497,3747497,(2016 EF195),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,25.50,{'kilometers': {'estimated_diameter_min': 0.02...,False,"[{'close_approach_date': '2020-01-01', 'close_...",False,NaN
4,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3893737,3893737,(2019 WE5),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,23.30,{'kilometers': {'estimated_diameter_min': 0.05...,False,"[{'close_approach_date': '2020-01-01', 'close_...",False,NaN


## 📊 Step 5: Quick Data Summary

Let's take a step back and see what we've collected.

In [10]:
print("\n" + "="*60)
print("📊 DATA COLLECTION SUMMARY")
print("="*60)

print(f"📡 Close Approaches: {len(ca_df):,} records")
print(f"📡 Sentry Objects: {len(sentry_df):,} records")
print(f"📡 NeoWs Data: {len(neows_df):,} records")

total_data_points = len(ca_df) + len(sentry_df) + len(neows_df)
print(f"\n📈 Total data points: {total_data_points:,}")
print(f"💾 Raw data saved to: data/raw/")

print("\n" + "="*60)


📊 DATA COLLECTION SUMMARY
📡 Close Approaches: 89,227 records
📡 Sentry Objects: 1,998 records
📡 NeoWs Data: 245,280 records

📈 Total data points: 336,505
💾 Raw data saved to: data/raw/



## 📈 Step 6: Basic Visualization

Let's make some quick charts to see what we're dealing with.

In [11]:
ca_df['approach_date'] = pd.to_datetime(ca_df['cd'])
ca_df['year'] = ca_df['approach_date'].dt.year

approaches_per_year = ca_df.groupby('year').size().reset_index(name='count')

fig = px.bar(
    approaches_per_year,
    x='year',
    y='count',
    title='📅 Asteroid Close Approaches Over Time (2020-2100)',
    labels={'year': 'Year', 'count': 'Number of Close Approaches'},
    color='count',
    color_continuous_scale='Reds'
)

fig.update_layout(
    template='plotly_dark',
    height=500,
    showlegend=False
)

fig.show()

print(f"\n📊 Peak year: {approaches_per_year.loc[approaches_per_year['count'].idxmax(), 'year']:.0f}")
print(f"📈 Max approaches in a year: {approaches_per_year['count'].max():,}")


📊 Peak year: 2022
📈 Max approaches in a year: 3,969


In [12]:
ca_df['dist_numeric'] = pd.to_numeric(ca_df['dist'], errors='coerce')

fig = px.histogram(
    ca_df,
    x='dist_numeric',
    nbins=50,
    title='🌍 How Close Did They Get? (Distance Distribution)',
    labels={'dist_numeric': 'Distance (AU)', 'count': 'Frequency'},
    color_discrete_sequence=['#FF6B6B']
)

fig.update_layout(
    template='plotly_dark',
    height=500,
    showlegend=False
)

fig.add_vline(
    x=0.05, 
    line_dash="dash", 
    line_color="yellow",
    annotation_text="Too close for comfort (0.05 AU)"
)

fig.show()

too_close = (ca_df['dist_numeric'] < 0.05).sum()
print(f"\n⚠️  Asteroids that came within 0.05 AU: {too_close:,}")
print(f"   (That's closer than Mercury to the Sun. Yikes.)")


⚠️  Asteroids that came within 0.05 AU: 18,881
   (That's closer than Mercury to the Sun. Yikes.)


## ✅ Step 7: Should You Panic Today?

Let's answer the most important question.

In [13]:
today = pd.Timestamp.now()
ca_df['days_until'] = (ca_df['approach_date'] - today).dt.days

upcoming = ca_df[ca_df['days_until'] > 0].sort_values('days_until')

if len(upcoming) > 0:
    next_approach = upcoming.iloc[0]
    
    print("\n" + "="*60)
    print("🎯 NEXT CLOSE CALL")
    print("="*60)
    print(f"🪨 Asteroid: {next_approach['des']}")
    print(f"📅 Date: {next_approach['cd']}")
    print(f"⏰ Days until approach: {next_approach['days_until']:.0f}")
    print(f"📏 Distance: {next_approach['dist_numeric']:.4f} AU")
    print(f"🚀 Velocity: {next_approach['v_rel']} km/s")
    
    distance = float(next_approach['dist_numeric'])
    
    if distance < 0.01:
        verdict = "😰 Worth monitoring"
    elif distance < 0.05:
        verdict = "😐 Interesting but safe"
    else:
        verdict = "😎 We're totally fine"
    
    print(f"\n💭 Verdict: {verdict}")
    print("\n✅ Should you panic today? Nah, we're good.")
    print("   (But we'll keep watching)\n")
    print("="*60)
else:
    print("\n✅ No upcoming close approaches in our dataset!")
    print("   Should you panic? Definitely not.")


🎯 NEXT CLOSE CALL
🪨 Asteroid: 2025 SO2
📅 Date: 2025-Oct-20 12:23
⏰ Days until approach: 1
📏 Distance: 0.0840 AU
🚀 Velocity: 10.3181833521213 km/s

💭 Verdict: 😎 We're totally fine

✅ Should you panic today? Nah, we're good.
   (But we'll keep watching)



## 🎯 Key Takeaways

### What We Learned:

1. **We collected A LOT of asteroid data** - thousands of close approaches
2. **Most asteroids pass safely** - distances are usually > 0.05 AU
3. **Sentry tracks the concerning ones** - but "concerning" is still very unlikely
4. **The data is rich** - we have dates, distances, velocities, orbital elements

### What's Next:

- **Notebook 02:** Clean and explore this data in depth
- **Process the data:** Merge all 3 sources into one unified dataset
- **Calculate risk scores:** Build our own panic level calculator
- **Upload to Kaggle:** Share this doom data with the world

### Bottom Line:

**Should you panic today?** No.

**Should you backup your data anyway?** Always a good idea.

---

*Remember: Looking up is optional. The data isn't.*

## 💾 Step 8: Save Your Work

The raw data is already saved by our fetcher script, but let's save processed versions too.

In [14]:
print("💾 Saving processed data for next notebook...")

Path('../data/processed').mkdir(parents=True, exist_ok=True)

ca_df.to_parquet('../data/processed/close_approaches.parquet')
sentry_df.to_parquet('../data/processed/sentry_objects.parquet')
neows_df.to_parquet('../data/processed/neows_data.parquet')

print("✅ Data saved!")
print("📁 Location: data/processed/")
print("\n📝 Next: Open 02_exploring_armageddon.ipynb")

💾 Saving processed data for next notebook...
✅ Data saved!
📁 Location: data/processed/

📝 Next: Open 02_exploring_armageddon.ipynb
